# SomaTrack Data Cleaning Pipeline

**Objective:** Transform raw survey responses into a consistent, correct, and complete dataset.

**Scope:** This notebook covers structural formatting, missing value imputation, outlier handling, and text standardization. It does **not** include mathematical transformations, scaling, or feature encoding. The output of this notebook is a pristine dataset ready for feature engineering.

In [13]:
import pandas as pd
import numpy as np

# Load the raw dataset from the Excel file
file_name = 'SomaTrack_—_Study_Habits_&_Physical_Health_Survey_Responses.xlsx'
df = pd.read_excel(file_name)

# Display the initial state
print(f"Initial shape: {df.shape}")
df.head()

Initial shape: (241, 33)


,daily_study_hours,study_days_per_week,longest_sitting_duration,study_break_frequency,study_break_duration,leave_desk_during_breaks,daily_water_intake,caffeine_intake_frequency,daily_screen_time,stress_level,...,tension_headache_frequency,wrist_pain_frequency,eye_strain_frequency,finger_numbness_frequency,physical_discomfort_level,age,gender,institution_type,field_of_study,year_of_study
0,12.0,7,4.0,Every 1-2 hours,5 - 10 minutes,Sometimes,1-1.5L,+2 drinks per day,4.0,High,...,1 — Mild / occasional (once or twice),1 — Mild / occasional (once or twice),2 — Moderate / regular (a few times a week),1 — Mild / occasional (once or twice),2 — Frequent discomfort (affects my focus),22.0,Female,Public University,Medical & Health Sciences,4th year
1,2.0,5,2.0,Every 30-60 minutes,10 - 30 minutes,"Yes, I walk or move around or just lie down on...",1-1.5L,Never,3.0,High,...,3 — Frequent / chronic (almost daily),2 — Moderate / regular (a few times a week),2 — Moderate / regular (a few times a week),1 — Mild / occasional (once or twice),2 — Frequent discomfort (affects my focus),23.0,Female,Other,Medical & Health Sciences,5th year
2,2.0,7,2.0,Every 1-2 hours,More than 30 minutes,"Yes, I walk or move around or just lie down on...",More than 2L,1–2 times per week,6.0,Moderate,...,3 — Frequent / chronic (almost daily),0 — Never,3 — Frequent / chronic (almost daily),3 — Frequent / chronic (almost daily),3 — Chronic pain (affects my daily life),20.0,Female,National Higher School (École Superieure),Computer Science & Artificial Intelligence,3rd year
3,6.0,6,3.0,Every 1-2 hours,10 - 30 minutes,"Yes, I walk or move around or just lie down on...",1-1.5L,3–5 times per week,6.0,Moderate,...,1 — Mild / occasional (once or twice),0 — Never,2 — Moderate / regular (a few times a week),0 — Never,1 — Mild / occasional discomfort,20.0,Male,National Higher School (École Superieure),Computer Science & Artificial Intelligence,3rd year
4,6.0,4,2.0,Every 30-60 minutes,5 - 10 minutes,"Yes, I walk or move around or just lie down on...",1-1.5L,Never,12.0,Moderate,...,2 — Moderate / regular (a few times a week),0 — Never,0 — Never,0 — Never,1 — Mild / occasional discomfort,21.0,Male,National Higher School (École Superieure),Computer Science & Artificial Intelligence,3rd year


## Step 1: Structural Integrity & Deduplication
First, we will ensure the column names are clean and remove any duplicate survey submissions to prevent individual users from skewing the dataset.

In [14]:
# Strip any accidental whitespace from column names
df.columns = df.columns.str.strip()

# Drop exact duplicate rows
df = df.drop_duplicates()
print(f"Shape after removing duplicates: {df.shape}")

Shape after removing duplicates: (241, 33)


## Step 2: Numeric Data Enforcement & Outlier Handling
Based on the survey rules, open-ended questions (like Q1, Q3, Q9, and Age) should be purely numeric. We need to convert these columns to proper float/integer types and handle impossible values.

**Identified Issues in Raw Data:**
* Floating-point anomalies (e.g., an age of `16.000001`)
* Impossible study hours (e.g., `1e-20` or inputs exceeding 24 hours)

In [15]:
# Define numeric columns based on the data dictionary
numeric_cols = [
    'daily_study_hours',
    'study_days_per_week',
    'longest_sitting_duration',
    'daily_screen_time',
    'age'
]

# Force conversion to numeric, turning text errors into NaN
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# 1. Clean Age: Round floating point ages to whole numbers
df['age'] = df['age'].round()

# 2. Clean Study Hours & Screen Time: Cap at 24 hours
df.loc[df['daily_study_hours'] > 24, 'daily_study_hours'] = 24
df.loc[df['daily_screen_time'] > 24, 'daily_screen_time'] = 24

# 3. Clean Study Days: Cap at 7 days
df.loc[df['study_days_per_week'] > 7, 'study_days_per_week'] = 7

# 4. Handle extreme minimum anomalies (e.g., the 1e-20 value)
df.loc[df['daily_study_hours'] < 0.1, 'daily_study_hours'] = 0

print(df[numeric_cols].describe())

       daily_study_hours  study_days_per_week  longest_sitting_duration  \
count         241.000000           241.000000                241.000000   
mean            4.711203             4.742739                  3.273997   
std             2.876746             1.640693                  2.731825   
min             0.000000             1.000000                  0.000000   
25%             3.000000             4.000000                  2.000000   
50%             4.000000             5.000000                  3.000000   
75%             6.000000             6.000000                  4.000000   
max            24.000000             7.000000                 17.000000   

       daily_screen_time         age  
count         241.000000  234.000000  
mean            7.168050   19.508547  
std             2.935021    1.548200  
min             1.000000   16.000000  
25%             5.000000   18.000000  
50%             7.000000   20.000000  
75%             9.000000   20.000000  
max         

## Step 3: Categorical Rule Enforcement
The survey had strict multiple-choice options. Text inputs often contain slight formatting variations (like trailing spaces or smart quotes) that cause the computer to treat the same answer as two different categories.

We will strip whitespace and standardize punctuation so the feature engineer receives perfectly uniform labels. Note: For Q17 (Pre-existing conditions), users could select multiple answers. We will clean the string but leave the comma-separated values intact—splitting these into binary columns is a feature engineering task.

In [16]:
# Select all categorical string columns
categorical_cols = df.select_dtypes(include=['object']).columns

# Standardize text: Strip leading/trailing whitespaces
for col in categorical_cols:
    df[col] = df[col].astype(str).str.strip()

# Fix specific punctuation inconsistencies observed in the data
# Example: "I don’t take breaks" (smart quote) vs "I don't take breaks" (straight quote)
df['leave_desk_during_breaks'] = df['leave_desk_during_breaks'].replace(
    "I don’t take breaks", "I don't take breaks"
)

# Standardize NaN representations in text columns (turning the string 'nan' back to an actual empty value)
df[categorical_cols] = df[categorical_cols].replace('nan', np.nan)

print("Categorical cleaning complete. Here are the unique values for the breaks question as a quick check:")
print(df['leave_desk_during_breaks'].unique())

Categorical cleaning complete. Here are the unique values for the breaks question as a quick check:
['Sometimes'
 'Yes, I walk or move around or just lie down on a bed or sofa'
 'No, I stay seated' "I don't take breaks"]


## Step 4: Logical Consistency Checks (Removing "Troll" Data)
Before imputing missing values, we need to drop survey respondents who submitted logically impossible or highly contradictory answers. These rows indicate the user either clicked randomly or did not read the questions carefully, meaning their physical pain metrics are unreliable.

We will flag and remove rows that trigger any of the following logical paradoxes:
1. **The Zero-Study Paradox:** Claiming 0 hours of daily study, but > 0 days of study per week.
2. **The Break Paradox:** Claiming an "I don't take breaks" break duration, while simultaneously claiming a regular break frequency.
3. **The Discomfort Paradox:** Claiming '0 — No pain at all' for general discomfort, while specifically reporting 'Moderate' or 'Frequent' pain in the neck, back, wrists, eyes, or hands.

In [17]:
# Rule 1: The Zero-Study Paradox
rule1 = (df['daily_study_hours'] == 0) & (df['study_days_per_week'] > 0)

# Rule 2: The Break Paradox
# Because we cleaned the text in Step 3, "I don't take breaks" is perfectly uniform
rule2 = (df['study_break_duration'] == "I don't take breaks") & (df['study_break_frequency'] != 'Never')

# Rule 3: The Discomfort Paradox
# Create a list of all specific pain columns to check
pain_cols = [
    'back_pain_frequency', 'neck_pain_frequency', 'tension_headache_frequency',
    'wrist_pain_frequency', 'eye_strain_frequency', 'finger_numbness_frequency'
]

# Build a dynamic mask that checks if ANY of those columns contain "2 — Moderate" or "3 — Frequent"
import pandas as pd
high_pain_mask = pd.Series([False]*len(df), index=df.index)
for col in pain_cols:
    high_pain_mask = high_pain_mask | df[col].astype(str).str.contains('3 — Frequent|2 — Moderate', na=False)

rule3 = (df['physical_discomfort_level'] == '0 — No pain at all') & high_pain_mask

# Combine all rules using the OR operator (|) to find any row that breaks AT LEAST one rule
bad_data_mask = rule1 | rule2 | rule3

print(f"Found {bad_data_mask.sum()} contradictory responses.")

# Drop the bad rows by keeping only the rows that do NOT (~) trigger the mask
df = df[~bad_data_mask]

print(f"New shape after dropping illogical data: {df.shape}")

Found 16 contradictory responses.
New shape after dropping illogical data: (225, 33)


## Step 4: Handling Missing Data
A clean dataset cannot contain missing values (`NaN`).
* **Numeric columns:** We will impute missing values using the median to avoid skewing from extreme outliers.
* **Categorical columns:** We will fill missing text fields with the label "Unknown".

In [18]:
# Check initial missing values
print("Missing values before handling:\n", df.isnull().sum()[df.isnull().sum() > 0])

# Impute Numeric missing values with the median
for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())

# Impute Categorical missing values with "Unknown"
for col in categorical_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna("Unknown")

# Verify no missing values remain
assert df.isnull().sum().sum() == 0, "Warning: There are still missing values!"
print("\nMissing values successfully handled.")

Missing values before handling:
 stress_level        1
backpack_weight     1
age                 6
gender              5
institution_type    4
field_of_study      6
year_of_study       9
dtype: int64

Missing values successfully handled.


## Step 5: Final Sanity Check and Export
The data is now structurally consistent, numeric types are properly formatted, categorical labels match the survey rules, impossible paradoxes have been removed, and there are no missing gaps.

The dataset is ready to be handed off. We will export it in two formats:
1. **CSV:** For the feature engineering team to load into their ML pipeline.
2. **Excel (XLSX):** For easy human-readable viewing and sharing.

In [19]:
# Final review of the data types and non-null counts
df.info()

# Export the pristine dataset to CSV
df.to_csv('SomaTrack_Cleaned_For_Feature_Engineering.csv', index=False)

# Export the pristine dataset to Excel
df.to_excel('SomaTrack_Cleaned_For_Feature_Engineering.xlsx', index=False)

print("\nClean dataset saved successfully to both CSV and Excel formats. Ready for handoff!")

<class 'pandas.core.frame.DataFrame'>
Index: 225 entries, 0 to 240
Data columns (total 33 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   daily_study_hours                      225 non-null    float64
 1   study_days_per_week                    225 non-null    int64  
 2   longest_sitting_duration               225 non-null    float64
 3   study_break_frequency                  225 non-null    object 
 4   study_break_duration                   225 non-null    object 
 5   leave_desk_during_breaks               225 non-null    object 
 6   daily_water_intake                     225 non-null    object 
 7   caffeine_intake_frequency              225 non-null    object 
 8   daily_screen_time                      225 non-null    float64
 9   stress_level                           225 non-null    object 
 10  study_location                         225 non-null    object 
 11  seat_type  